## 0 Omnilingual Baseline

This notebook gets the baseline Word Error Rate (WER) and Character Error Rate (CER) scores for Meta's Omnilingual model.

In [1]:
import torch
torch.backends.cudnn.enabled = False
from tqdm import tqdm
from omnilingual_asr.data import load_all_data
from omnilingual_asr.evaluate import add_metrics_columns, idiom_summary, print_evaluation_summary, show_examples
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs
from omnilingual_asr.utils import get_best_gpu, normalize_romansh_text

/local/scratch/matuor/Romansh-ASR/omnilingual/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


First we set the configuration, the `MODEL_CARD` variable defines the model that we want to evaluate.

In [2]:
MODEL_CARD = "omniASR_LLM_1B_v2"
LANGUAGE_CODE = "roh_Latn_surs1244"
BATCH_SIZE = 8
best_gpu = get_best_gpu()
DEVICE = f"cuda:{best_gpu}" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}, Lang supported: {LANGUAGE_CODE in supported_langs}")

Selected GPU 0 with 24088 MiB free memory
Device: cuda:0, Lang supported: True


Then we load the test data from `data/clean-data`.

In [3]:
df_test = load_all_data("test")
print(f"Loaded {len(df_test)} samples")

Loaded 631 samples


Then we can start transcribing the test set.

In [4]:
pipeline = ASRInferencePipeline(model_card=MODEL_CARD, device=DEVICE)
audio_paths = df_test["audio_path"].tolist()
transcriptions = []

for i in tqdm(range(0, len(audio_paths), BATCH_SIZE)):
    batch = audio_paths[i:i+BATCH_SIZE]
    try:
        results = pipeline.transcribe(batch, lang=[LANGUAGE_CODE]*len(batch), batch_size=len(batch))
        transcriptions.extend(results)
    except Exception as e:
        print(f"Batch error at {i}: {e}")
        transcriptions.extend([""] * len(batch))

df_test["omnilingual_transcription"] = transcriptions

/local/scratch/matuor/Romansh-ASR/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/local/scratch/matuor/Romansh-ASR/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/local/scratch/matuor/Romansh-ASR/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

 56%|█████▌    | 44/79 [12:12<07:38, 13.11s/it]

Batch error at 344: The map function has failed while processing the path 'data' of the input data. See nested exception for details.


100%|██████████| 79/79 [23:12<00:00, 17.63s/it]


With the transcriptions we can compute the Word Error Rate (WER) and Character Error Rate (CER).

In [5]:
df_test['omnilingual_transcription'] = df_test['omnilingual_transcription'].apply(normalize_romansh_text)
df_test = add_metrics_columns(df_test, "sentence", "omnilingual_transcription")
summary = idiom_summary(df_test)
print_evaluation_summary(summary)


OVERALL RESULTS
Total test samples: 631
Word Error Rate (WER): 54.39%
Character Error Rate (CER): 18.17%

PER IDIOM RESULTS

CC-2021-05-28
  Samples: 81
  WER: 34.35%
  CER: 9.80%

RMPUTER-CC-2021-06-11
  Samples: 114
  WER: 56.52%
  CER: 19.20%

RMSURSILV-CC-2021-05-28
  Samples: 94
  WER: 40.89%
  CER: 13.62%

RMSURSIV-CC-2021-12-23
  Samples: 151
  WER: 62.66%
  CER: 21.35%

RMSUTSILV-CC-2022-05-18
  Samples: 94
  WER: 71.69%
  CER: 24.56%

RMVALLADER-CC-2021-05-28
  Samples: 97
  WER: 52.78%
  CER: 17.50%

SUMMARY TABLE
                   idiom  samples  wer_mean  wer_std  cer_mean  cer_std
           cc-2021-05-28       81     34.35    10.87      9.80     6.02
   rmputer-cc-2021-06-11      114     56.52    18.69     19.20    11.28
 rmsursilv-cc-2021-05-28       94     40.89    14.90     13.62     6.75
  rmsursiv-cc-2021-12-23      151     62.66    18.38     21.35    11.78
 rmsutsilv-cc-2022-05-18       94     71.69    10.64     24.56     7.40
rmvallader-cc-2021-05-28       97    

Finally we can show some example transcriptions.

In [12]:
show_examples(df_test, "omnilingual_transcription")


EXAMPLE TRANSCRIPTIONS

Idiom: rmputer-cc-2021-06-11
Reference:    dasper ils murs vegnan eir churos ils chastagners actuelmaing dad una gruppa da recruts civils chi appredschan la lavur our illa natura
Hypothesis:   daspera ls murs vegnen er ciuros als ciastagners actuelmeinc da d una grupa de recruts zivils ci apregian la lavur ori la natura
WER: 77.3% | CER: 16.3%
----------------------------------------

Idiom: cc-2021-05-28
Reference:    e deblas vischnancas chen mo pli posts executivs dal chantun il referendum duai esser in cler signal cunter adina dapli prescripziuns da surengiu e pe
Hypothesis:   e deblas vishnanchas chen mopli posts executivs dal chantun il referendum duei esser in cler signal cunter adina da pli per scripziuns dasurengiu e pe
WER: 26.3% | CER: 4.3%
----------------------------------------

Idiom: rmsursilv-cc-2021-05-28
Reference:    memia tard vegnevan plitost quels che eran datier dalla scola nus eran adina ad uras nun chei deva in incaps chei era giu lavi